In [1]:
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment

In [2]:
df = pd.read_csv("/Users/jonathanzhu/nematostella_videos/imgs/rpa_bb_16C_9_dpf_celldish_04_02_2026/positions.csv", index_col=0)
df.head()

,image_filename,x1,y1,x2,y2,conf,class
0,/Users/jonathanzhu/nematostella_videos/imgs/rp...,459.16450,600.93000,485.07820,612.59000,0.957247,1.0
1,/Users/jonathanzhu/nematostella_videos/imgs/rp...,772.72614,484.01890,788.78710,506.91350,0.954870,1.0
2,/Users/jonathanzhu/nematostella_videos/imgs/rp...,760.34620,541.13890,773.05290,551.60410,0.934535,0.0
3,/Users/jonathanzhu/nematostella_videos/imgs/rp...,197.46588,686.65485,214.34150,698.35130,0.918712,1.0
4,/Users/jonathanzhu/nematostella_videos/imgs/rp...,546.09340,634.06280,565.46716,645.74567,0.914138,1.0


In [3]:
#adding attributes to dataframe based on positions
df["width"] = df["x2"] - df["x1"]
df["height"] = df["y2"] - df["y1"]
df["x_center"] = df["x2"] - 0.5 * df["width"]
df["y_center"] = df["y2"] - 0.5 * df["height"]
df.head()

,image_filename,x1,y1,x2,y2,conf,class,width,height,x_center,y_center
0,/Users/jonathanzhu/nematostella_videos/imgs/rp...,459.16450,600.93000,485.07820,612.59000,0.957247,1.0,25.91370,11.66000,472.12135,606.760000
1,/Users/jonathanzhu/nematostella_videos/imgs/rp...,772.72614,484.01890,788.78710,506.91350,0.954870,1.0,16.06096,22.89460,780.75662,495.466200
2,/Users/jonathanzhu/nematostella_videos/imgs/rp...,760.34620,541.13890,773.05290,551.60410,0.934535,0.0,12.70670,10.46520,766.69955,546.371500
3,/Users/jonathanzhu/nematostella_videos/imgs/rp...,197.46588,686.65485,214.34150,698.35130,0.918712,1.0,16.87562,11.69645,205.90369,692.503075
4,/Users/jonathanzhu/nematostella_videos/imgs/rp...,546.09340,634.06280,565.46716,645.74567,0.914138,1.0,19.37376,11.68287,555.78028,639.904235


In [4]:
#helper functions for making a cost matrix from two dataframes
def euclidean_distance(x1, y1, x2, y2):
    return np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)

def cost_matrix(df_first, df_second):
    cm = np.zeros((len(df_first), len(df_second)))
    for i in range(len(df_first)):
        for j in range(len(df_second)):
            x1 = df_first.iloc[i]["x_center"]
            y1 = df_first.iloc[i]["y_center"]
            x2 = df_second.iloc[j]["x_center"]
            y2 = df_second.iloc[j]["y_center"]

            cm[i, j] = euclidean_distance(x1, y1, x2, y2)
    
    return cm


In [5]:
filenames = df["image_filename"].unique()
filenames = np.sort(filenames) #sorting only the filenames makes this a lot quicker
df_individuals = []

for f in filenames:
    df_individuals.append(df[df.image_filename == f])

In [6]:
df_lens = [len(d) for d in df_individuals]
df_lens

[21,
 21,
 22,
 22,
 22,
 23,
 21,
 23,
 22,
 23,
 24,
 23,
 22,
 23,
 22,
 21,
 23,
 22,
 21,
 21,
 22,
 24,
 21,
 23,
 22,
 22,
 24,
 24,
 26,
 25,
 26,
 24,
 25,
 24,
 26,
 26,
 22,
 24,
 23,
 23,
 22,
 21,
 22,
 21,
 23,
 22,
 23,
 22,
 23,
 23,
 23,
 23,
 22,
 24,
 22,
 22,
 23,
 22,
 22,
 22,
 21,
 22,
 21,
 23,
 22,
 22,
 22,
 23,
 22,
 21,
 24,
 22,
 22,
 23,
 22,
 22,
 23,
 22,
 21,
 22,
 23,
 23,
 22,
 22,
 21,
 21,
 22,
 22,
 21,
 22,
 21,
 22,
 22,
 22,
 21,
 22,
 21,
 22,
 22,
 22,
 22,
 22,
 21,
 23,
 21,
 22,
 22,
 22,
 22,
 22,
 22,
 22,
 22,
 23,
 23,
 22,
 22,
 21,
 22,
 22,
 22,
 23,
 21,
 21,
 25,
 20,
 19,
 20,
 20,
 21,
 21,
 19,
 20,
 20,
 21,
 20,
 20,
 21,
 20,
 19,
 24,
 21,
 21,
 19,
 20,
 20,
 19,
 22,
 22,
 21,
 22,
 24,
 24,
 23,
 21,
 22,
 21,
 21,
 21,
 20,
 21,
 22,
 22,
 22,
 23,
 21,
 21,
 24,
 22,
 21,
 21,
 23,
 21,
 21,
 24,
 23,
 26,
 25,
 24,
 21,
 21,
 21,
 21,
 21,
 22,
 22,
 22,
 22,
 22,
 25,
 24,
 23,
 22,
 21,
 21,
 23,
 22,
 21,
 23,
 21,


In [7]:
cm_test = cost_matrix(df_individuals[0], df_individuals[1])

In [8]:
row_ind, col_ind = linear_sum_assignment(cm_test)
row_ind, col_ind

(array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
        17, 18, 19, 20]),
 array([ 0,  1,  2,  3,  4, 13,  6,  7,  8,  9,  5, 10, 12, 11, 14, 16, 17,
        15, 18, 19, 20]))

In [9]:
cm_test[row_ind, col_ind].sum()

np.float64(4.478241426117113)

In [10]:
df_individuals[0]["id"] = row_ind
df_individuals[0]

/var/folders/px/b7vc3nh913zb_m0x36ncftj00000gn/T/ipykernel_59352/3598560501.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_individuals[0]["id"] = row_ind


,image_filename,x1,y1,x2,y2,conf,class,width,height,x_center,y_center,id
14318,/Users/jonathanzhu/nematostella_videos/imgs/rp...,460.58115,597.905400,484.39760,611.28890,0.965658,1.0,23.81645,13.383500,472.489375,604.597150,0
14319,/Users/jonathanzhu/nematostella_videos/imgs/rp...,408.76650,554.336550,428.40707,577.71080,0.946368,1.0,19.64057,23.374250,418.586785,566.023675,1
14320,/Users/jonathanzhu/nematostella_videos/imgs/rp...,768.65967,486.155240,784.61220,508.71630,0.927595,1.0,15.95253,22.561060,776.635935,497.435770,2
14321,/Users/jonathanzhu/nematostella_videos/imgs/rp...,191.31021,748.115970,201.84398,759.77850,0.925966,0.0,10.53377,11.662530,196.577095,753.947235,3
14322,/Users/jonathanzhu/nematostella_videos/imgs/rp...,780.16550,742.135700,790.54443,753.94340,0.916117,0.0,10.37893,11.807700,785.354965,748.039550,4
14323,/Users/jonathanzhu/nematostella_videos/imgs/rp...,798.67210,603.470500,809.30780,613.33484,0.910040,0.0,10.63570,9.864340,803.989950,608.402670,5
14324,/Users/jonathanzhu/nematostella_videos/imgs/rp...,547.78660,633.035460,565.38560,644.77770,0.904606,1.0,17.59900,11.742240,556.586100,638.906580,6
14325,/Users/jonathanzhu/nematostella_videos/imgs/rp...,256.78064,628.348000,268.12580,641.74493,0.900534,0.0,11.34516,13.396930,262.453220,635.046465,7
14326,/Users/jonathanzhu/nematostella_videos/imgs/rp...,834.36786,555.327940,843.77010,564.81335,0.897065,0.0,9.40224,9.485410,839.068980,560.070645,8
14327,/Users/jonathanzhu/nematostella_videos/imgs/rp...,796.70820,769.324800,809.80630,780.63684,0.892304,0.0,13.09810,11.312040,803.257250,774.980820,9


In [ ]:
#helper function: take the ids of one dataframe, then assign corresponding ones to the second dataframe
#based on row and column indices (all given as parameters)
def assign_ids(df_first, df_second, row_ind, col_ind):
    new_ids = np.full(len(df_second), -1)

    #this routine handles if df_first has the same number of entries as df_second. 
    if len(df_first) <= len(df_second):
        #go through the IDs in the first df
        for i in range(len(df_first)):
            #get the ID itself
            current_id = df_first.iloc[i]["id"]

            #get the corresponding index from the row index array
            j = np.where(row_ind == i)[0][0]

            #get the corresponding index from the columns
            k = np.where(col_ind == j)[0][0]

            #assign the same id to index k of second dataframe
            new_ids[k] = current_id

        df_second["id"] = new_ids

    #case in which df_first has more entries than df_second.
    elif len(df_first) > len(df_second):
        ids_first = [id for id in range(len(df_first))]
        used_ids = []
        #same iteration as above but with some slight changes
        for i in range(len(df_second)):
            #get the ID itself
            current_id = df_first.iloc[i]["id"]
            used_ids.append(current_id)

            #get the corresponding index from the row index array
            j = np.where(row_ind == i)[0][0]

            #get the corresponding index from the columns
            k = np.where(col_ind == j)[0][0]

            #assign the same id to index k of second dataframe
            new_ids[k] = current_id

        unused_ids = [id for id in ids_first if id not in used_ids]
        for x in unused_ids:
            pd.concat(df_second, df_first[df_first.id == x])
            
        df_second["image_filename"] = df_second.iloc[0]["image_filename"]

    elif len(df_first) < len(df_second):
        print("todo")
        df_second = df_second[df_second.id != -1]
    
    else:
        print("Something went horribly wrong")
        assert False


In [12]:
assign_ids(df_individuals[0], df_individuals[1], row_ind, col_ind)
df_individuals[1]

/var/folders/px/b7vc3nh913zb_m0x36ncftj00000gn/T/ipykernel_59352/2662064369.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_second["id"] = new_ids


,image_filename,x1,y1,x2,y2,conf,class,width,height,x_center,y_center,id
11867,/Users/jonathanzhu/nematostella_videos/imgs/rp...,460.62918,598.00230,484.44376,611.44147,0.965828,1.0,23.81458,13.43917,472.536470,604.721885,0
11868,/Users/jonathanzhu/nematostella_videos/imgs/rp...,408.73780,554.37660,428.39273,577.79150,0.944150,1.0,19.65493,23.41490,418.565265,566.084050,1
11869,/Users/jonathanzhu/nematostella_videos/imgs/rp...,768.62850,486.29214,784.50684,509.04956,0.929161,1.0,15.87834,22.75742,776.567670,497.670850,2
11870,/Users/jonathanzhu/nematostella_videos/imgs/rp...,191.31836,748.15010,201.84941,759.81160,0.925300,0.0,10.53105,11.66150,196.583885,753.980850,3
11871,/Users/jonathanzhu/nematostella_videos/imgs/rp...,779.94500,742.93780,790.36566,755.07750,0.916791,0.0,10.42066,12.13970,785.155330,749.007650,4
11872,/Users/jonathanzhu/nematostella_videos/imgs/rp...,558.17710,886.12450,569.86260,897.34040,0.905866,0.0,11.68550,11.21590,564.019850,891.732450,13
11873,/Users/jonathanzhu/nematostella_videos/imgs/rp...,547.88040,633.08057,565.33636,644.78064,0.905287,1.0,17.45596,11.70007,556.608380,638.930605,6
11874,/Users/jonathanzhu/nematostella_videos/imgs/rp...,256.79410,628.43726,268.12802,641.80570,0.901088,0.0,11.33392,13.36844,262.461060,635.121480,7
11875,/Users/jonathanzhu/nematostella_videos/imgs/rp...,834.42000,555.43225,843.82610,564.89670,0.898382,0.0,9.40610,9.46445,839.123050,560.164475,8
11876,/Users/jonathanzhu/nematostella_videos/imgs/rp...,796.67365,769.34690,809.81380,780.77924,0.887138,0.0,13.14015,11.43234,803.243725,775.063070,9


In [18]:
cm_test = cost_matrix(df_individuals[1], df_individuals[2])
row_ind, col_ind = linear_sum_assignment(cm_test)
print(row_ind, col_ind)
assign_ids(df_individuals[1], df_individuals[2], row_ind, col_ind)
df_individuals[2] = df_individuals[2][df_individuals[2].id != -1]
df_individuals[2]

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20] [ 0  1  2  3  4  6  5  7  8 10  9 13 12 11 14 16 15 17 18 19 20]


,image_filename,x1,y1,x2,y2,conf,class,width,height,x_center,y_center,id
10097,/Users/jonathanzhu/nematostella_videos/imgs/rp...,460.70035,597.95920,484.48413,611.30080,0.965376,1.0,23.78378,13.34160,472.592240,604.630000,0
10098,/Users/jonathanzhu/nematostella_videos/imgs/rp...,408.78140,554.32010,428.42007,577.73505,0.944429,1.0,19.63867,23.41495,418.600735,566.027575,1
10099,/Users/jonathanzhu/nematostella_videos/imgs/rp...,768.71240,486.25270,784.55945,508.50485,0.927997,1.0,15.84705,22.25215,776.635925,497.378775,2
10100,/Users/jonathanzhu/nematostella_videos/imgs/rp...,191.34386,748.06647,201.84721,759.65985,0.926358,0.0,10.50335,11.59338,196.595535,753.863160,3
10101,/Users/jonathanzhu/nematostella_videos/imgs/rp...,779.57336,744.38983,790.29175,756.91070,0.921960,0.0,10.71839,12.52087,784.932555,750.650265,4
10102,/Users/jonathanzhu/nematostella_videos/imgs/rp...,547.76290,633.02466,565.44700,644.79950,0.907354,1.0,17.68410,11.77484,556.604950,638.912080,6
10103,/Users/jonathanzhu/nematostella_videos/imgs/rp...,559.71270,884.72910,570.87744,896.06630,0.906136,0.0,11.16474,11.33720,565.295070,890.397700,13
10104,/Users/jonathanzhu/nematostella_videos/imgs/rp...,256.76715,628.35126,268.12503,641.75134,0.902918,0.0,11.35788,13.40008,262.446090,635.051300,7
10105,/Users/jonathanzhu/nematostella_videos/imgs/rp...,834.44490,555.33960,843.87427,564.92065,0.901452,0.0,9.42937,9.58105,839.159585,560.130125,8
10106,/Users/jonathanzhu/nematostella_videos/imgs/rp...,691.77040,865.68414,703.11420,876.61365,0.884742,0.0,11.34380,10.92951,697.442300,871.148895,5


In [17]:
cm_test = cost_matrix(df_individuals[2], df_individuals[3])
row_ind, col_ind = linear_sum_assignment(cm_test)
print(row_ind, col_ind)
assign_ids(df_individuals[2], df_individuals[3], row_ind, col_ind)
df_individuals[3] = df_individuals[3][df_individuals[3].id != -1]
df_individuals[3]

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20] [ 0  1  2  3  9  6  4  8  7 10 11  5 13 12 14 17 15 16 18 19 21]


IndexError: index 0 is out of bounds for axis 0 with size 0